In [ ]:
import pandas as pd
import numpy as np
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
tab = pd.read_csv('/home/shared/splus_gaia/data/h-alpha-selection-marina/MC/halpha_emitters_mc_simbad_and_all.csv')
len(tab)

In [ ]:
tab = tab[(tab['mag_psf_r'] < 19.5) & (tab['gaia_ruwe'] < 1.6)]

In [ ]:
len(tab)

In [ ]:
tab['main_type'].unique()#i want a list of the kind of types

In [ ]:
tab['halpha_types'].unique()

In [ ]:
from astroquery.gaia import Gaia
import pandas as pd
import numpy as np
from pathlib import Path

INPUT = '/home/shared/splus_gaia/data/h-alpha-selection-marina/MC/halpha_emitters_mc_simbad_and_all.csv'
OUTPUT = '/home/shared/splus_gaia/data/h-alpha-selection-marina/MC/halpha_emitters_mc_simbad_and_all_x_gaia_espels.csv'
BATCH  = 10_000  # tune if needed

# 1) Load and get unique source_ids
#    Make sure the column name matches your CSV EXACTLY:
ID_COL = "gaia_source_id"

df = pd.read_csv(INPUT, dtype={ID_COL: "Int64"})
all_ids = (df[ID_COL]
           .dropna()
           .astype("int64")
           .unique())

if all_ids.size == 0:
    pd.DataFrame().to_csv(OUTPUT, index=False)
    print(f"No IDs found in column '{ID_COL}'. Wrote empty file: {OUTPUT}")
else:
    out_parts = []
    for i in range(0, len(all_ids), BATCH):
        ids = tuple(all_ids[i:i+BATCH].tolist())  # Python tuple renders as (1, 2, 3)
        adql = f"""
        SELECT
          source_id,
          ew_espels_halpha,
          ew_espels_halpha_uncertainty, 
          ew_espels_halpha_flag, 
          ew_espels_halpha_model, 
          classlabel_espels, 
          classlabel_espels_flag, 
          classprob_espels_wcstar, 
          classprob_espels_wnstar, 
          classprob_espels_bestar, 
          classprob_espels_ttauristar, 
          classprob_espels_herbigstar, 
          classprob_espels_dmestar, 
          classprob_espels_pne
        FROM gaiadr3.astrophysical_parameters
        WHERE source_id IN {ids}
        """
        job = Gaia.launch_job_async(adql)
        out_parts.append(job.get_results().to_pandas())

    out = pd.concat(out_parts, ignore_index=True) if out_parts else pd.DataFrame()
    out.to_csv(OUTPUT, index=False)
    print(f"Saved: {OUTPUT}  | rows={len(out)}")

In [ ]:
out

In [ ]:
tab = pd.merge(tab, out, left_on='gaia_source_id', right_on='source_id', how='left')

In [ ]:
halpha_map = {
    # 1. Hot Hydrogen Recombination Emitters
    'Be*':'Hot_HII_Emitters','Be*_Candidate':'Hot_HII_Emitters',
    'Ae*':'Hot_HII_Emitters','Ae*_Candidate':'Hot_HII_Emitters',
    'WolfRayet*':'Hot_HII_Emitters','BlueSG':'Hot_HII_Emitters',
    'BlueSG_Candidate':'Hot_HII_Emitters','Supergiant':'Hot_HII_Emitters',
    'HIIReg':'Hot_HII_Emitters','StarFormingReg':'Hot_HII_Emitters',
    'EmLine*':'Hot_HII_Emitters','EmObj':'Hot_HII_Emitters',

    # 2. Accretion / Binary Systems
    'CataclyV*':'Accretion_Binary','CataclyV*_Candidate':'Accretion_Binary',
    'Symbiotic*':'Accretion_Binary','XrayBin':'Accretion_Binary',
    'XrayBin_Candidate':'Accretion_Binary','HighMassXBin':'Accretion_Binary',
    'HighMassXBin_Candidate':'Accretion_Binary','Nova':'Accretion_Binary',
    'Supernova':'Accretion_Binary','Supernova_Candidate':'Accretion_Binary',
    'EclBin':'Accretion_Binary','EllipVar':'Accretion_Binary',

    # 3. Evolved / Nebular Emitters
    'PlanetaryNeb':'Evolved_Nebular','PlanetaryNeb_Candidate':'Evolved_Nebular',
    'post-AGB*':'Evolved_Nebular','post-AGB*_Candidate':'Evolved_Nebular',
    'LongPeriodV*':'Evolved_Nebular','LongPeriodV*_Candidate':'Evolved_Nebular',
    'Mira':'Evolved_Nebular','AGB*':'Evolved_Nebular','AGB*_Candidate':'Evolved_Nebular',
    'C*':'Evolved_Nebular','C*_Candidate':'Evolved_Nebular','S*':'Evolved_Nebular',
    'RVTauV*':'Evolved_Nebular','RCrBV*':'Evolved_Nebular',

    # 4. Young / Pre-MS Emitters
    'YSO':'PreMS_Accretion','YSO_Candidate':'PreMS_Accretion','OrionV*':'PreMS_Accretion',
    'HIIReg':'PreMS_Accretion','StarFormingReg':'PreMS_Accretion','EmObj':'PreMS_Accretion',

    # 5. Extragalactic H-alpha Sources
    'Galaxy':'Extragalactic_Halpha','EmissionG':'Extragalactic_Halpha',
    'AGN':'Extragalactic_Halpha','AGN_Candidate':'Extragalactic_Halpha',
    'QSO':'Extragalactic_Halpha','QSO_Candidate':'Extragalactic_Halpha',
    'Seyfert1':'Extragalactic_Halpha','Seyfert2':'Extragalactic_Halpha',
    'RadioG':'Extragalactic_Halpha','InteractingG':'Extragalactic_Halpha',
    'ClG':'Extragalactic_Halpha','ClG_Candidate':'Extragalactic_Halpha',
    'BrightestCG':'Extragalactic_Halpha'
}

tab['halpha_types'] = tab['main_type'].map(halpha_map)

In [ ]:
tab

In [ ]:
tab.to_csv('/home/shared/splus_gaia/data/h-alpha-selection-marina/MC/halpha_emitters_mc_simbad_and_all.csv')

In [ ]:
simbad = pd.read_csv('/home/shared/splus_gaia/data/h-alpha-selection-marina/MC/halpha_emitters_mc_simbad.csv')

In [ ]:
simbad['iJ660'] = simbad['mag_psf_i'] - simbad['mag_psf_j0660']
simbad['660_excess'] = simbad['mag_psf_j0660'] - (simbad['mag_psf_i'] - simbad['mag_psf_r'])/2

In [ ]:
len(simbad)

In [ ]:
#filter simbad table to only have the 5 top ocurences in the main types
top5_types = simbad['main_type'].value_counts().nlargest(10).index.tolist()
simbad_top5 = simbad[simbad['main_type'].isin(top5_types[1:])]

In [ ]:
import matplotlib.pyplot as plt

# Plot por tipo (cada tipo com cor própria) e legenda legível
types = simbad_top5['main_type'].unique()
cmap = plt.get_cmap('tab20')
colors = {t: cmap(i % cmap.N) for i, t in enumerate(types)}

for t in types:
    subset = simbad_top5[simbad_top5['main_type'] == t]
    if subset.empty:
        continue # Fundo cinza claro
    plt.scatter(subset['ri'], subset['rJ0660'], s=2, alpha=0.5, color=colors[t], label=t)
    

# Ajustes da legenda: vários tipos em colunas e fora do gráfico para não sobrepor
plt.legend(markerscale=3, fontsize='small', ncol=2, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlabel('rJ0660')
plt.ylabel('ri')

plt.title('simbad_top5: rJ0660 vs ri por main_type')
plt.tight_layout()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

def plot_halpha_diagrams(df, type_col='halpha_types', show='all'):
    """
    Plot H-alpha related color-color and 3D diagrams using S-PLUS magnitudes.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame with columns like 'mag_psf_r', 'mag_psf_j0660', etc.
    type_col : str
        Column name for the emission-type grouping.
    show : str
        One of {'2d', '3d', 'plotly', 'all'} defining which plots to show.
    """

    # ---------- helpers ----------
    def mag_col(band: str) -> str:
        b = band.strip().lower()
        # aceita j660/j0660/J0660
        if b.startswith('j') and len(b) in (4, 5) and b[1:].isdigit():
            if len(b) == 4:  # j660 -> j0660
                b = 'j0' + b[1:]
        return f'mag_psf_{b}'

    def color_idx_from_bands(a: str, b: str) -> pd.Series:
        ca, cb = mag_col(a), mag_col(b)
        if ca not in df.columns or cb not in df.columns:
            raise KeyError(f"Missing column(s): '{ca}' or '{cb}' not in DataFrame.")
        return df[ca] - df[cb]

    # ---------- 1) índices 2D pré-computados (nomes com sublinhado) ----------
    color_defs = {
        'r_i': ('r', 'i'),
        'r_j0660': ('r', 'j0660'),
        'j0515_j0660': ('j0515', 'j0660'),
        'j0660_i': ('j0660', 'i'),
        'j0660_j0861': ('j0660', 'j0861'),
        'u_j0378': ('u', 'j0378'),
        'j0378_j0410': ('j0378', 'j0410'),
        'g_r': ('g', 'r'),
        'j0395_j0410': ('j0395', 'j0410'),
        'i_j0861': ('i', 'j0861'),
        'j0515_r': ('j0515', 'r'),
        'j0430_g': ('j0430', 'g')
    }
    for cname, (a, b) in color_defs.items():
        ca, cb = f'mag_psf_{a.lower()}', f'mag_psf_{b.lower()}'
        if ca in df.columns and cb in df.columns:
            df[cname] = df[ca] - df[cb]

    # ---------- 2) pares 2D e triplets 3D ----------
    diagrams_2d = [
        ('r_i', 'r_j0660'),
        ('j0515_j0660', 'j0660_i'),
        ('r_i', 'j0660_j0861'),
        ('u_j0378', 'j0378_j0410'),
        ('g_r', 'r_j0660'),
        ('j0395_j0410', 'j0660_j0861'),
        ('i_j0861', 'j0660_j0861'),
        ('j0515_r', 'j0660_j0861'),
        ('r_j0660', 'j0660_j0861'),
        ('j0430_g', 'r_j0660')
    ]

    # triplets como expressões 'A - B'
    triplets_3d = [
        ('g - r', 'r - J0660', 'J0660 - J0861'),
        ('r - i', 'r - J0660', 'J0660 - J0861'),
        ('J0515 - J0660', 'J0660 - i', 'g - r'),
        ('u - J0378', 'J0378 - J0410', 'r - J0660'),
        ('J0395 - J0410', 'J0660 - J0861', 'r - i'),
        ('J0515 - r', 'J0660 - J0861', 'J0430 - g'),
    ]

    # ---------- 3) paleta ----------
    types = df[type_col].dropna().unique()
    cmap = plt.get_cmap('tab20')
    colors = {t: cmap(i % cmap.N) for i, t in enumerate(sorted(types))}

    # ---------- 4) 2D (matplotlib) ----------
    if show in ['2d', 'all']:
        ncols = 3
        nrows = int(np.ceil(len(diagrams_2d) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(15, 12))
        axes = axes.flatten()

        for i, (x, y) in enumerate(diagrams_2d):
            ax = axes[i]
            # checa se os índices existem
            if x not in df.columns or y not in df.columns:
                ax.set_visible(False)
                continue
            for t in types:
                subset = df[df[type_col] == t]
                if subset.empty:
                    continue
                ax.scatter(subset[x], subset[y], s=6, alpha=0.5, color=colors[t], label=t)
            ax.set_xlabel(x); ax.set_ylabel(y)
            ax.set_title(f'{y} vs {x}', fontsize=9)
            ax.grid(True, alpha=0.3)

        # legenda única
        handles, labels = axes[0].get_legend_handles_labels()
        if handles:
            fig.legend(handles, labels, loc='upper right', ncol=2, fontsize='small', markerscale=3)
        plt.tight_layout(rect=[0, 0, 0.85, 1])
        plt.show()

    # ---------- 5) 3D estático (matplotlib) ----------
    if show in ['3d', 'all']:
        ncols = 3
        nrows = int(np.ceil(len(triplets_3d) / ncols))
        fig = plt.figure(figsize=(16, 10))

        for k, (x_def, y_def, z_def) in enumerate(triplets_3d, 1):
            # normaliza traços
            x_def = x_def.replace('–', '-').replace('—', '-')
            y_def = y_def.replace('–', '-').replace('—', '-')
            z_def = z_def.replace('–', '-').replace('—', '-')

            try:
                Ax, Bx = [s.strip() for s in x_def.split('-')]
                Ay, By = [s.strip() for s in y_def.split('-')]
                Az, Bz = [s.strip() for s in z_def.split('-')]
            except ValueError:
                raise ValueError(f"Triplet '{x_def}, {y_def}, {z_def}' malformed — use 'A - B'.")

            # compute arrays (não acessa df['g - r']!)
            try:
                x_vals = color_idx_from_bands(Ax, Bx).values
                y_vals = color_idx_from_bands(Ay, By).values
                z_vals = color_idx_from_bands(Az, Bz).values
            except KeyError as e:
                # cria subplot vazio e avisa no título
                ax = fig.add_subplot(nrows, ncols, k, projection='3d')
                ax.set_title(f"Missing column: {e}", fontsize=8)
                ax.set_axis_off()
                continue

            ax = fig.add_subplot(nrows, ncols, k, projection='3d')
            for t in types:
                mask = (df[type_col] == t)
                if not np.any(mask):
                    continue
                ax.scatter(x_vals[mask], y_vals[mask], z_vals[mask],
                           s=6, alpha=0.45, color=colors[t], depthshade=True, label=t)
            ax.set_xlabel(x_def); ax.set_ylabel(y_def); ax.set_zlabel(z_def)
            ax.set_title(f'{z_def} vs {y_def} vs {x_def}', fontsize=9)
            ax.view_init(elev=22, azim=35)

        # legenda única
        if len(fig.axes) > 0:
            handles, labels = [], []
            H, L = fig.axes[0].get_legend_handles_labels()
            handles.extend(H); labels.extend(L)
            if handles:
                fig.legend(handles, labels, loc='upper right', ncol=2,
                           fontsize='small', markerscale=3)
        plt.tight_layout(rect=[0, 0, 0.86, 1])
        plt.show()

    # ---------- 6) 3D interativo (Plotly) ----------
    if show in ['plotly', 'all']:
        palette = (px.colors.qualitative.Dark24 +
                   px.colors.qualitative.Plotly +
                   px.colors.qualitative.Set3)
        color_map = {t: palette[i % len(palette)] for i, t in enumerate(sorted(types))}

        for (x_def, y_def, z_def) in triplets_3d:
            x_def = x_def.replace('–', '-').replace('—', '-')
            y_def = y_def.replace('–', '-').replace('—', '-')
            z_def = z_def.replace('–', '-').replace('—', '-')

            try:
                Ax, Bx = [s.strip() for s in x_def.split('-')]
                Ay, By = [s.strip() for s in y_def.split('-')]
                Az, Bz = [s.strip() for s in z_def.split('-')]
            except ValueError:
                raise ValueError(f"Triplet '{x_def}, {y_def}, {z_def}' malformed — use 'A - B'.")

            # monta df_plot calculando cores na hora
            try:
                df_plot = pd.DataFrame({
                    'x': color_idx_from_bands(Ax, Bx),
                    'y': color_idx_from_bands(Ay, By),
                    'z': color_idx_from_bands(Az, Bz),
                    type_col: df[type_col].values
                }).replace([np.inf, -np.inf], np.nan).dropna(subset=['x', 'y', 'z'])
            except KeyError as e:
                # pula este triplet se faltar coluna
                print(f"[plotly] skipping '{x_def}, {y_def}, {z_def}': {e}")
                continue

            title = f'{z_def} vs {y_def} vs {x_def}'
            fig = px.scatter_3d(
                df_plot,
                x='x', y='y', z='z',
                color=type_col,
                color_discrete_map=color_map,
                opacity=0.6,
                title=title,
                labels={'x': x_def, 'y': y_def, 'z': z_def}
            )
            fig.update_traces(marker=dict(size=3), selector=dict(mode='markers'))
            fig.update_layout(
                legend=dict(itemsizing='constant', bgcolor='rgba(255,255,255,0.7)'),
                margin=dict(l=0, r=0, t=40, b=0),
                scene=dict(
                    xaxis=dict(title=x_def, showspikes=False),
                    yaxis=dict(title=y_def, showspikes=False),
                    zaxis=dict(title=z_def, showspikes=False),
                    camera=dict(eye=dict(x=1.7, y=1.4, z=1.1))
                )
            )
            fig.show()

In [ ]:
plot_halpha_diagrams(simbad_top5, 'main_type', show = 'all')

In [ ]:
classif = pd.read_csv('/home/shared/splus_gaia/data/h-alpha-selection-marina/classified_final.csv')

plot_halpha_diagrams(classif, 'AstroInspectClass', show = 'all')

In [ ]:
simbad.groupby('main_type')['id'].count()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns  # só para paleta de cores agradável

plt.figure(figsize=(8,6))

# -----------------------------
# 1) Scatter base - todos os objetos
# -----------------------------
plt.scatter(
    simbad['mag_psf_r'], simbad['mu_max_r'],
    s=6, alpha=0.25, color='lightgray', label='All sources', zorder=1
)

# -----------------------------
# 2) Cálculo e plot do "stellar ridge"
# -----------------------------
bins = np.arange(simbad['mag_psf_r'].min(), simbad['mag_psf_r'].max(), 0.5)
med = simbad.groupby(pd.cut(simbad['mag_psf_r'], bins))['mu_max_r'].median().dropna()
bin_centers = bins[:-1] + 0.25
plt.plot(
    bin_centers[:len(med)], med,
    color='black', lw=2.2, label='Stellar ridge (median)', zorder=3
)

# -----------------------------
# 3) Destacar os 5 principais tipos (main_type)
# -----------------------------
# Obtém os 5 tipos mais frequentes
top_types = simbad['main_type'].value_counts().nlargest(6).index[1:6]

palette = sns.color_palette("Set1", len(top_types))

for color, t in zip(palette, top_types):
    subset = simbad[simbad['main_type'] == t]
    plt.scatter(
        subset['mag_psf_r'], subset['mu_max_r'],
        s=14, alpha=0.8, label=t, color=color,
        edgecolor='black', linewidth=0.3, zorder=2
    )

# -----------------------------
# 4) Eixos, título e legendas
# -----------------------------
plt.xlabel('MAG_PSF_r', fontsize=12)
plt.ylabel('MU_MAX_r [mag arcsec$^{-2}$]', fontsize=12)
plt.title('μ_max vs. r-band magnitude — Point-source identification', fontsize=13, pad=10)

plt.gca().invert_yaxis()   # estrelas mais brilhantes (menor μ) no topo
plt.grid(alpha=0.3, linestyle='--')

# Legenda: posicionada fora do gráfico para clareza
plt.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), title='Main types', frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

bands = ['u', 'J0378', 'J0395', 'J0410', 'J0430', 'g', 'J0515', 'r', 'J0660', 'i', 'J0861', 'z']
types = emline['main_type'].unique()

# Defina número de colunas e linhas no grid por objeto
ncols = 4
nrows = int(np.ceil(len(bands) / ncols))

for obj_type in types:
    subset = emline[emline['main_type'] == obj_type]
    
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(18, 10))
    axes = axes.flatten()
    
    for i, band in enumerate(bands):
        ax = axes[i]
        ax.hist(subset[f'mu_max_{band.lower()}'], bins=20, alpha=0.7, color='steelblue')
        ax.set_title(f'{band}', fontsize=10)
        ax.set_xlabel('Magnitude')
        ax.set_ylabel('Count')
    
    # Apagar eixos extras se a grade tiver mais espaços que bandas
    for j in range(len(bands), len(axes)):
        fig.delaxes(axes[j])
    
    fig.suptitle(f'Magnitude Distribution for {obj_type}', fontsize=16)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

In [ ]:
halpha_map_10 = {
    # 1. Hot Emission-Line Stars
    'Be*':'Hot_Emission_Stars','Be*_Candidate':'Hot_Emission_Stars',
    'Ae*':'Hot_Emission_Stars','Ae*_Candidate':'Hot_Emission_Stars',
    'WolfRayet*':'Hot_Emission_Stars','BlueSG':'Hot_Emission_Stars',
    'BlueSG_Candidate':'Hot_Emission_Stars','Supergiant':'Hot_Emission_Stars',

    # 2. Massive HII / Ionized Regions
    'HIIReg':'HII_Regions','StarFormingReg':'HII_Regions',
    'EmLine*':'HII_Regions','EmObj':'HII_Regions',

    # 3. Young Stellar Objects
    'YSO':'YSO_PMS','YSO_Candidate':'YSO_PMS','OrionV*':'YSO_PMS',

    # 4. Symbiotic & Interacting Binaries
    'Symbiotic*':'Symbiotic_Binaries','post-AGB*_Candidate':'Symbiotic_Binaries',

    # 5. Cataclysmic / Accreting Compact Systems
    'CataclyV*':'Accreting_Compact','CataclyV*_Candidate':'Accreting_Compact',
    'XrayBin':'Accreting_Compact','XrayBin_Candidate':'Accreting_Compact',
    'HighMassXBin':'Accreting_Compact','HighMassXBin_Candidate':'Accreting_Compact',
    'Nova':'Accreting_Compact',

    # 6. Eruptive / Transient
    'Supernova':'Eruptive_Transient','Supernova_Candidate':'Eruptive_Transient',
    'Eruptive*':'Eruptive_Transient','Transient':'Eruptive_Transient',

    # 7. Evolved Cool Giants
    'AGB*':'Evolved_Cool_Giants','AGB*_Candidate':'Evolved_Cool_Giants',
    'LongPeriodV*':'Evolved_Cool_Giants','LongPeriodV*_Candidate':'Evolved_Cool_Giants',
    'Mira':'Evolved_Cool_Giants','C*':'Evolved_Cool_Giants','C*_Candidate':'Evolved_Cool_Giants',
    'S*':'Evolved_Cool_Giants','RVTauV*':'Evolved_Cool_Giants','RCrBV*':'Evolved_Cool_Giants',

    # 8. Planetary Nebulae & post-AGB Shells
    'PlanetaryNeb':'Planetary_Nebulae','PlanetaryNeb_Candidate':'Planetary_Nebulae',
    'post-AGB*':'Planetary_Nebulae','RGB*':'Planetary_Nebulae',

    # 9. Normal Stellar / Weak Emitters
    'Star':'Normal_Stars','Variable*':'Normal_Stars','PulsV*':'Normal_Stars',
    'Cepheid':'Normal_Stars','ClassicalCep':'Normal_Stars','Type2Cep':'Normal_Stars',
    'RRLyrae':'Normal_Stars','RRLyrae_Candidate':'Normal_Stars','EllipVar':'Normal_Stars',
    'HighPM*':'Normal_Stars','ChemPec*':'Normal_Stars','RGB*_Candidate':'Normal_Stars',

    # 10. Extragalactic H-alpha Sources
    'Galaxy':'Extragalactic_Sources','EmissionG':'Extragalactic_Sources',
    'AGN':'Extragalactic_Sources','AGN_Candidate':'Extragalactic_Sources',
    'Seyfert1':'Extragalactic_Sources','Seyfert2':'Extragalactic_Sources',
    'QSO':'Extragalactic_Sources','QSO_Candidate':'Extragalactic_Sources',
    'RadioG':'Extragalactic_Sources','InteractingG':'Extragalactic_Sources',
    'ClG':'Extragalactic_Sources','ClG_Candidate':'Extragalactic_Sources',
    'BrightestCG':'Extragalactic_Sources'
}

tab['halpha_class10'] = tab['main_type'].map(halpha_map_10).fillna('Unclassified')

In [ ]:
plot_halpha_diagrams(tab, type_col='halpha_class10', show='all')

In [ ]:
# number of distinct halpha_class10 groups
tab.groupby('halpha_class10').size().count(
)